# Sen1Floods11: Exploratory Data Analysis & Radiometric Verification
### B.M.S. College of Engineering | Department of Computer Science & Engineering
**UG 5th-Semester Mini-Project**  
*Anchor Milestone: Review-2 (Design with Basic Module Implementation)*

---
### Objectives:
1. Inspect raw Sentinel-1 SAR dual-polarization GeoTIFF chips (`S1Hand`: VV and VH bands).
2. Verify empirical backscatter values to confirm data is already calibrated in decibels (dB).
3. Inspect ground truth hand-labeled masks (`LabelHand`: `-1` = ignore, `0` = dry land, `1` = water).
4. Quantify class imbalance to prove why overall accuracy is misleading for flood mapping.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import rasterio
import matplotlib.pyplot as plt

print("Rasterio Version:", rasterio.__version__)
print("NumPy Version:", np.__version__)

## 1. Radiometric Range Verification (dB vs Linear)
A common pitfall is assuming SAR values are linear and mistakenly applying $10 \cdot \log_{10}(\sigma^0)$.  
Let us inspect the actual numerical distribution of raw pixel values in both VV and VH bands.

In [ ]:
s1_path = Path("../data/raw/S1Hand/Bolivia_103757_S1Hand.tif")
lbl_path = Path("../data/raw/LabelHand/Bolivia_103757_LabelHand.tif")

with rasterio.open(s1_path) as src:
    s1 = src.read()  # (2, 512, 512)
    s1_meta = src.meta

with rasterio.open(lbl_path) as src:
    lbl = src.read(1)  # (512, 512)

vv = s1[0]
vh = s1[1]

print("S1 Metadata:", s1_meta)
print(f"VV Band: Min = {np.nanmin(vv):.2f} dB, Max = {np.nanmax(vv):.2f} dB, Mean = {np.nanmean(vv):.2f} dB")
print(f"VH Band: Min = {np.nanmin(vh):.2f} dB, Max = {np.nanmax(vh):.2f} dB, Mean = {np.nanmean(vh):.2f} dB")
print(f"Unique Label Values: {np.unique(lbl)}")

**Finding:** The negative values (roughly $[-45, 0]$ dB) empirically confirm that `S1Hand` chips are **already expressed in decibels (dB)**.  
Applying an additional $\log_{10}$ would corrupt the data with NaNs. The correct preprocessing is clipping to valid physical thresholds followed by Z-score standardization.

In [ ]:
# Plot Backscatter Histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(vv[np.isfinite(vv)].flatten(), bins=100, color='royalblue', alpha=0.8)
axes[0].set_title("Sentinel-1 VV Backscatter Distribution (dB)")
axes[0].set_xlabel("Decibels (dB)")
axes[0].set_ylabel("Pixel Count")
axes[0].grid(True, alpha=0.3)

axes[1].hist(vh[np.isfinite(vh)].flatten(), bins=100, color='darkorange', alpha=0.8)
axes[1].set_title("Sentinel-1 VH Backscatter Distribution (dB)")
axes[1].set_xlabel("Decibels (dB)")
axes[1].set_ylabel("Pixel Count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Multi-Panel Visual Inspection

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# VV
im0 = axes[0].imshow(vv, cmap='gray', vmin=-30, vmax=0)
axes[0].set_title("SAR VV (dB)")
axes[0].axis('off')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# VH
im1 = axes[1].imshow(vh, cmap='gray', vmin=-35, vmax=-5)
axes[1].set_title("SAR VH (dB)")
axes[1].axis('off')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# False-color composite: R=VV, G=VH, B=VV/VH
vv_norm = np.clip((vv + 30) / 30, 0, 1)
vh_norm = np.clip((vh + 35) / 30, 0, 1)
ratio = np.clip(vv - vh, 0, 15) / 15
composite = np.stack([vv_norm, vh_norm, ratio], axis=-1)
axes[2].imshow(composite)
axes[2].set_title("False-Color SAR Composite")
axes[2].axis('off')

# Ground Truth
lbl_vis = np.zeros((*lbl.shape, 3))
lbl_vis[lbl == 0] = [0.1, 0.1, 0.1]    # Land (Dark)
lbl_vis[lbl == 1] = [0.0, 0.8, 1.0]    # Water (Cyan)
lbl_vis[lbl == -1] = [1.0, 0.8, 0.0]   # Ignore (Yellow)
axes[3].imshow(lbl_vis)
axes[3].set_title("Ground Truth (Cyan=Water, Yellow=Ignore)")
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 3. Class Imbalance Quantification
Why overall accuracy is misleading: Non-flooded land dominates terrestrial scenes.

In [ ]:
flood_pixels = int(np.sum(lbl == 1))
land_pixels = int(np.sum(lbl == 0))
ignore_pixels = int(np.sum(lbl == -1))
valid_pixels = flood_pixels + land_pixels
total_pixels = valid_pixels + ignore_pixels

print("=" * 50)
print("CLASS DISTRIBUTION ANALYSIS:")
print("=" * 50)
print(f"Total Pixels:        {total_pixels:,}")
print(f"Valid Pixels:        {valid_pixels:,} ({valid_pixels/total_pixels*100:.1f}%)")
print(f"Ignore Pixels (-1):  {ignore_pixels:,} ({ignore_pixels/total_pixels*100:.1f}%)")
print(f"Flood Pixels (1):    {flood_pixels:,} ({flood_pixels/valid_pixels*100:.2f}% of valid)")
print(f"Dry Land Pixels (0): {land_pixels:,} ({land_pixels/valid_pixels*100:.2f}% of valid)")
print("=" * 50)
print(f"Trap Baseline: A model predicting 0 everywhere achieves {land_pixels/valid_pixels*100:.2f}% accuracy,")
print("but fails completely on disaster response (0.00% Flood IoU)!")